# 03B — Image & Text Embedding Extraction

**Rubric targets:** Model Development (Neural Networks)

**Syllabus coverage:**
- **Week 6 (CNNs):** ResNet-50 image embeddings — transfer learning, convolutional architecture
- **Week 8 (Transformers):** Sentence-transformer text embeddings — attention, transformer architecture

This notebook extracts dense vector representations of articles from two modalities:
1. **Product images** → 2048-d embeddings via ResNet-50 (pretrained on ImageNet)
2. **Text descriptions** → 384-d embeddings via all-MiniLM-L6-v2 (pretrained sentence transformer)

These embeddings will be used as features in the LightGBM ranker on Day 4.
The key evaluation question: **do deep learning representations add value over
hand-engineered features?**

**Run this notebook on a GPU runtime.** Image extraction takes ~45-60 minutes.
Run 03A (collaborative filtering) in another tab while this processes.

In [7]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT = '/content/drive/MyDrive/MLII_Final'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [8]:
import os

sold_articles = set(pd.read_parquet(f'{PROJECT}/data/split/train.parquet')['article_id'].unique())
folders_needed = sorted(set(aid[:3] for aid in sold_articles))

os.makedirs('/content/images_local', exist_ok=True)

for folder in folders_needed:
    !cp -r "{PROJECT}/data/images/{folder}" /content/images_local/

print(f"Done — copied {len(folders_needed)} folders")

Done — copied 86 folders


In [9]:
!pip install -q sentence-transformers

In [10]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image
from pathlib import Path
from tqdm import tqdm
import time

DATA_DIR = Path(f'{PROJECT}/data/parquet')
OUT_DIR = Path(f'{PROJECT}/outputs')
OUT_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Device: cuda
GPU: Tesla T4


In [11]:
articles = pd.read_parquet(DATA_DIR / 'articles.parquet')
print(f"Articles: {len(articles):,}")

Articles: 105,542


---
## Part 1: Image Embeddings (ResNet-50)

### Architecture Choice

**ResNet-50** is a 50-layer deep convolutional neural network that won the
2015 ImageNet competition. Its key architectural innovation is **residual
connections** (skip connections) that allow gradients to flow through the
network without vanishing, enabling training of much deeper networks than
was previously possible.

We use ResNet-50 pretrained on ImageNet (1.2M images, 1000 classes) as a
**feature extractor** — this is **transfer learning**. The idea: the lower
layers of the network learn general visual features (edges, textures, shapes)
that transfer well to new domains. The upper layers learn task-specific
features. By removing the final classification head, we extract a 2048-dimensional
embedding that encodes the visual "essence" of each product image.

**Why not fine-tune?** We don't have labeled pairs of "similar" products —
only purchase data. Fine-tuning would require a supervised signal that we
don't have. The pretrained features are sufficient for computing visual
similarity between items.

In [12]:
# Load ResNet-50 pretrained on ImageNet, remove classification head
resnet = models.resnet50(pretrained=True)
# Remove the final fully-connected layer
# ResNet-50 architecture: conv layers → AdaptiveAvgPool → FC(2048→1000)
# We keep everything up to the pooling layer → output is 2048-d
resnet = nn.Sequential(*list(resnet.children())[:-1])  # remove FC
resnet = resnet.to(device)
resnet.eval()

print("ResNet-50 loaded (pretrained ImageNet, classification head removed)")
print(f"Output: 2048-dimensional embedding per image")

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


ResNet-50 loaded (pretrained ImageNet, classification head removed)
Output: 2048-dimensional embedding per image


In [13]:
# ImageNet preprocessing transforms
# These match the preprocessing used during ResNet-50 training
preprocess = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],  # ImageNet means
        std=[0.229, 0.224, 0.225],    # ImageNet stds
    ),
])

In [14]:
# Custom dataset for article images
# H&M images are stored as: images/0{article_id[:3]}/0{article_id}.jpg
IMAGE_DIR = Path('/content/images_local')

class ArticleImageDataset(Dataset):
    def __init__(self, article_ids, image_dir, transform):
        self.article_ids = article_ids
        self.image_dir = image_dir
        self.transform = transform

        # Build paths and check which exist
        self.valid_items = []
        for aid in article_ids:
            # H&M image path convention: images/0{first 3 digits}/0{article_id}.jpg
            img_path = image_dir / aid[:3] / f'{aid}.jpg'
            if img_path.exists():
                self.valid_items.append((aid, img_path))

        print(f"Found images for {len(self.valid_items):,} / {len(article_ids):,} articles")

    def __len__(self):
        return len(self.valid_items)

    def __getitem__(self, idx):
        aid, path = self.valid_items[idx]
        try:
            img = Image.open(path).convert('RGB')
            img = self.transform(img)
        except Exception:
            # Return a blank image if loading fails
            img = torch.zeros(3, 224, 224)
        return aid, img

# Create dataset
dataset = ArticleImageDataset(
    article_ids=articles['article_id'].tolist(),
    image_dir=IMAGE_DIR,
    transform=preprocess,
)

dataloader = DataLoader(dataset, batch_size=64, shuffle=False, num_workers=0)

Found images for 105,100 / 105,542 articles


In [15]:
# Extract embeddings
print(f"Extracting image embeddings for {len(dataset):,} articles...")
t0 = time.time()

all_ids = []
all_embeddings = []

with torch.no_grad():
    for batch_ids, batch_imgs in tqdm(dataloader, desc="Image embeddings"):
        batch_imgs = batch_imgs.to(device)
        embeddings = resnet(batch_imgs).squeeze(-1).squeeze(-1)  # (batch, 2048)

        all_ids.extend(batch_ids)
        all_embeddings.append(embeddings.cpu().numpy())

image_embeddings = np.vstack(all_embeddings)
elapsed = time.time() - t0
print(f"\nDone: {image_embeddings.shape} in {elapsed:.0f}s ({elapsed/60:.1f} min)")

Extracting image embeddings for 105,100 articles...


Image embeddings: 100%|██████████| 1643/1643 [54:43<00:00,  2.00s/it]



Done: (105100, 2048) in 3284s (54.7 min)


In [16]:
# L2-normalize embeddings (makes cosine similarity = dot product)
norms = np.linalg.norm(image_embeddings, axis=1, keepdims=True)
norms[norms == 0] = 1  # avoid division by zero
image_embeddings_normed = image_embeddings / norms

# Save as parquet
img_emb_df = pd.DataFrame(
    image_embeddings_normed,
    columns=[f'img_emb_{i}' for i in range(image_embeddings_normed.shape[1])]
)
img_emb_df.insert(0, 'article_id', all_ids)

img_emb_df.to_parquet(OUT_DIR / 'image_embeddings.parquet', index=False)
print(f"✓ Saved image_embeddings.parquet: {img_emb_df.shape}")
print(f"  {len(all_ids):,} articles with embeddings")

✓ Saved image_embeddings.parquet: (105100, 2049)
  105,100 articles with embeddings


In [17]:
# Quick sanity check: nearest neighbors for a popular item
from sklearn.metrics.pairwise import cosine_similarity

# Pick a popular article
test_aid = all_ids[0]
test_emb = image_embeddings_normed[0:1]

# Cosine similarity against all items
sims = cosine_similarity(test_emb, image_embeddings_normed)[0]
top_indices = np.argsort(sims)[::-1][1:6]  # top 5 nearest (excluding self)

test_article_info = articles[articles['article_id'] == test_aid][['article_id', 'prod_name', 'colour_group_name', 'department_name']].iloc[0]
print(f"Query: {test_article_info['prod_name']} ({test_article_info['colour_group_name']}) — {test_article_info['department_name']}")
print(f"\nNearest neighbors by image embedding:")
for idx in top_indices:
    nn_aid = all_ids[idx]
    nn_info = articles[articles['article_id'] == nn_aid]
    if len(nn_info) > 0:
        nn_info = nn_info.iloc[0]
        print(f"  sim={sims[idx]:.4f}  {nn_info['prod_name']} ({nn_info['colour_group_name']}) — {nn_info['department_name']}")

Query: Strap top (Black) — Jersey Basic

Nearest neighbors by image embedding:
  sim=0.9775  PE Emma Camisole top 2 (Black) — Take Care External
  sim=0.9752  Nina Strap (Black) — Basic 1
  sim=0.9738  gaby (Black) — Jersey
  sim=0.9724  V-neck strap top (Black) — Jersey Basic
  sim=0.9680  Strap Top. (Black) — Jersey Basic


---
## Part 2: Text Embeddings (Sentence-Transformer)

### Architecture Choice

**all-MiniLM-L6-v2** is a lightweight sentence transformer — a transformer
encoder that maps variable-length text into a fixed 384-dimensional embedding
where semantically similar texts are close in vector space.

The model is based on the **transformer architecture** introduced in
"Attention is All You Need" (Vaswani et al., 2017). Key components:

- **Self-attention mechanism:** Each token attends to every other token in the
  input, learning which words are relevant to each other. This is what makes
  transformers powerful for understanding context.

- **Multi-head attention:** Multiple attention "heads" capture different types
  of relationships (syntactic, semantic, positional).

- **Positional encoding:** Since transformers process all tokens in parallel
  (unlike RNNs), positional information is injected via sinusoidal encodings.

The sentence-transformers library adds a **pooling layer** on top of the
token-level transformer outputs to produce a single sentence embedding.

**Why this model?** It's fast (~14K sentences/sec on GPU), the 384-d output
is compact, and it was trained on over 1 billion sentence pairs — making it
robust for our product descriptions.

In [18]:
from sentence_transformers import SentenceTransformer

text_model = SentenceTransformer('all-MiniLM-L6-v2', device=device.type)
print(f"Sentence transformer loaded: all-MiniLM-L6-v2")
print(f"Output: 384-dimensional embedding per text")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Sentence transformer loaded: all-MiniLM-L6-v2
Output: 384-dimensional embedding per text


In [19]:
# Prepare text descriptions
articles_with_desc = articles[articles['detail_desc'].notna()].copy()
print(f"Articles with descriptions: {len(articles_with_desc):,} / {len(articles):,}")

# For articles without descriptions, create a fallback from metadata
articles_no_desc = articles[articles['detail_desc'].isna()].copy()
articles_no_desc['detail_desc'] = (
    articles_no_desc['prod_name'].fillna('') + ' ' +
    articles_no_desc['product_type_name'].fillna('') + ' ' +
    articles_no_desc['colour_group_name'].fillna('') + ' ' +
    articles_no_desc['department_name'].fillna('')
).str.strip()
print(f"Articles using metadata fallback: {len(articles_no_desc):,}")

all_articles_text = pd.concat([articles_with_desc, articles_no_desc], ignore_index=True)
all_articles_text = all_articles_text.sort_values('article_id').reset_index(drop=True)

Articles with descriptions: 105,126 / 105,542
Articles using metadata fallback: 416


In [20]:
# Extract text embeddings
print(f"Extracting text embeddings for {len(all_articles_text):,} articles...")
t0 = time.time()

text_descriptions = all_articles_text['detail_desc'].tolist()
text_ids = all_articles_text['article_id'].tolist()

text_embeddings = text_model.encode(
    text_descriptions,
    batch_size=256,
    show_progress_bar=True,
    normalize_embeddings=True,  # L2 normalize for cosine sim
)

elapsed = time.time() - t0
print(f"\nDone: {text_embeddings.shape} in {elapsed:.0f}s ({elapsed/60:.1f} min)")

Extracting text embeddings for 105,542 articles...


Batches:   0%|          | 0/413 [00:00<?, ?it/s]


Done: (105542, 384) in 43s (0.7 min)


In [21]:
# Save as parquet
text_emb_df = pd.DataFrame(
    text_embeddings,
    columns=[f'text_emb_{i}' for i in range(text_embeddings.shape[1])]
)
text_emb_df.insert(0, 'article_id', text_ids)

text_emb_df.to_parquet(OUT_DIR / 'text_embeddings.parquet', index=False)
print(f"✓ Saved text_embeddings.parquet: {text_emb_df.shape}")

✓ Saved text_embeddings.parquet: (105542, 385)


In [22]:
# Sanity check: text nearest neighbors
test_idx = 0
test_text_aid = text_ids[test_idx]
test_text_emb = text_embeddings[test_idx:test_idx+1]

text_sims = cosine_similarity(test_text_emb, text_embeddings)[0]
top_text_indices = np.argsort(text_sims)[::-1][1:6]

test_info = articles[articles['article_id'] == test_text_aid].iloc[0]
print(f"Query: {test_info['prod_name']} — {test_info['detail_desc'][:80]}...")
print(f"\nNearest neighbors by text embedding:")
for idx in top_text_indices:
    nn_aid = text_ids[idx]
    nn_info = articles[articles['article_id'] == nn_aid]
    if len(nn_info) > 0:
        nn_info = nn_info.iloc[0]
        desc_preview = str(nn_info.get('detail_desc', ''))[:60]
        print(f"  sim={text_sims[idx]:.4f}  {nn_info['prod_name']} — {desc_preview}...")

Query: Strap top — Jersey top with narrow shoulder straps....

Nearest neighbors by text embedding:
  sim=1.0000  Strap top — Jersey top with narrow shoulder straps....
  sim=1.0000  Strap top (1) — Jersey top with narrow shoulder straps....
  sim=0.9629  Nina Strap — Fitted jersey top with narrow shoulder straps....
  sim=0.9629  Nina Strap — Fitted jersey top with narrow shoulder straps....
  sim=0.9629  Nina Strap — Fitted jersey top with narrow shoulder straps....


---
## Summary

**What we extracted:**

| Modality | Model | Architecture | Dimensions | Syllabus |
|----------|-------|-------------|------------|----------|
| Product images | ResNet-50 | CNN with residual connections | 2048 | Week 6: CNNs, transfer learning |
| Text descriptions | all-MiniLM-L6-v2 | Transformer encoder | 384 | Week 8: Transformers, attention, embeddings |

**Saved artifacts:**
- `outputs/image_embeddings.parquet` — L2-normalized 2048-d image embeddings
- `outputs/text_embeddings.parquet` — L2-normalized 384-d text embeddings

**How Day 4 will use these:**
For each (customer, candidate_item) pair in the ranker, compute:
1. **Image cosine similarity:** candidate embedding vs average of user's last 10 purchase embeddings
2. **Text cosine similarity:** candidate embedding vs average of user's purchased item embeddings

These become features in LightGBM alongside the hand-engineered features.
The embedding ablation on Day 4 will test whether these deep learning
representations add value beyond what purchase patterns and metadata capture.